# Device Health Report
This notebook visualizes device health status, sensor status, communication, data collection, and cloud connectivity, similar to the provided dashboard.

In [1]:
# Import required libraries
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

In [2]:
# Fetch device health data from Microsoft Defender API
import os
import requests
import pandas as pd
from IPython.display import display
from getpass import getpass

# Prompt for credentials if not already set in environment
def get_credentials():
    tenant_id = os.environ.get('MDE_TENANT_ID')
    client_id = os.environ.get('MDE_CLIENT_ID')
    client_secret = os.environ.get('MDE_CLIENT_SECRET')
    
    if not tenant_id:
        tenant_id = input("Enter your Tenant ID: ").strip()
        os.environ['MDE_TENANT_ID'] = tenant_id
        
    if not client_id:
        client_id = input("Enter your Client ID: ").strip()
        os.environ['MDE_CLIENT_ID'] = client_id
        
    if not client_secret:
        client_secret = getpass("Enter your Client Secret: ").strip()
        os.environ['MDE_CLIENT_SECRET'] = client_secret
    
    return tenant_id, client_id, client_secret

print("Please enter your Microsoft Defender for Endpoint API credentials:")
TENANT_ID, CLIENT_ID, CLIENT_SECRET = get_credentials()

if not all([CLIENT_ID, CLIENT_SECRET, TENANT_ID]):
    raise ValueError("All credentials (Tenant ID, Client ID, and Client Secret) are required.")

# Get OAuth2 token with WindowsDefenderATP scope
def get_token():
    url = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    data = {
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': 'https://api.securitycenter.windows.com/.default',
        'grant_type': 'client_credentials'
    }
    print("Requesting access token...")
    resp = requests.post(url, data=data)
    if not resp.ok:
        print(f"Token request failed with status {resp.status_code}")
        print(f"Error details: {resp.text}")
    resp.raise_for_status()
    return resp.json()['access_token']

try:
    token = get_token()
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json'
    }

    # Query Microsoft Defender API for devices
    url = 'https://api.securitycenter.windows.com/api/machines'
    print(f"Requesting data from: {url}")
    response = requests.get(url, headers=headers)
    print(f"API request status: {response.status_code}")
    
    if not response.ok:
        print(f"API request failed: {response.text}")
    response.raise_for_status()
    
    data = response.json()['value']
    df = pd.json_normalize(data)
    
    # Map the columns from Defender API to our dashboard format
    df['Health Status'] = df['healthStatus']
    df['Sensor'] = df['onboardingStatus'].map({'Onboarded': 'Enabled', 'NotOnboarded': 'Disabled'})
    df['Sensor Communication'] = df['healthStatus']  # Can be 'Active', 'Inactive'
    df['Sensor Data Collection'] = df['healthStatus']
    df['Cloud Connectivity'] = df['healthStatus']
    
    # Show available columns for debugging
    print("\nAvailable columns in the API response:")
    print(df.columns.tolist())
    
    display(df.head())

except Exception as e:
    print(f"Error occurred: {str(e)}")
    if 'response' in locals():
        print(f"Response content: {response.text}")

Please enter your Microsoft Defender for Endpoint API credentials:
Requesting access token...
Requesting access token...
Requesting data from: https://api.securitycenter.windows.com/api/machines
Requesting data from: https://api.securitycenter.windows.com/api/machines
API request status: 200

Available columns in the API response:
['id', 'mergedIntoMachineId', 'isPotentialDuplication', 'isExcluded', 'exclusionReason', 'computerDnsName', 'firstSeen', 'lastSeen', 'osPlatform', 'osProcessor', 'version', 'lastIpAddress', 'lastExternalIpAddress', 'agentVersion', 'osBuild', 'healthStatus', 'deviceValue', 'rbacGroupId', 'rbacGroupName', 'riskScore', 'exposureLevel', 'isAadJoined', 'aadDeviceId', 'machineTags', 'onboardingStatus', 'osArchitecture', 'managedBy', 'managedByStatus', 'ipAddresses', 'vmMetadata.vmId', 'vmMetadata.cloudProvider', 'vmMetadata.resourceId', 'vmMetadata.subscriptionId', 'Health Status', 'Sensor', 'Sensor Communication', 'Sensor Data Collection', 'Cloud Connectivity']
AP

,id,mergedIntoMachineId,isPotentialDuplication,isExcluded,exclusionReason,computerDnsName,firstSeen,lastSeen,osPlatform,osProcessor,...,ipAddresses,vmMetadata.vmId,vmMetadata.cloudProvider,vmMetadata.resourceId,vmMetadata.subscriptionId,Health Status,Sensor,Sensor Communication,Sensor Data Collection,Cloud Connectivity
0,07db4b6b265cc224f8ddc96cd549874b889b30d2,None,False,False,None,hybrid-vm,2025-08-21T23:02:02.2462477Z,2025-08-23T06:27:06.9829603Z,WindowsServer2022,x64,...,"[{'ipAddress': '10.0.2.4', 'macAddress': '000D...",5a25b8f4-866c-445f-a490-8fc7d7df4330,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,082909d8-d042-4ef9-99f9-dd4a4ed283de,Inactive,Enabled,Inactive,Inactive,Inactive
1,18f95523fefd1962c2ff7931021e566544f6414b,None,False,False,None,testwin2,2025-06-15T12:07:44.7228143Z,2025-06-15T12:07:44.7228143Z,Windows10,None,...,"[{'ipAddress': '10.0.0.5', 'macAddress': '0022...",7a8d0984-45f4-4a7a-a755-45c450ea9aa8,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,None,Active,NaN,Active,Active,Active
2,20f5c067141911591f2929516e3db31beaddef32,None,False,False,None,test123,2025-06-23T17:00:28.090865Z,2025-06-24T06:44:36.95051Z,Ubuntu,None,...,"[{'ipAddress': '10.1.0.4', 'macAddress': '6045...",121dec4e-9cfc-40b3-9299-743303db7418,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,None,Active,NaN,Active,Active,Active
3,3b43d0758eac3d033c81bb587b09dc76e1add42b,None,False,False,None,testmdevm,2025-04-22T15:38:28.590487Z,2025-06-03T06:19:36.712736Z,Other,None,...,"[{'ipAddress': '10.0.0.4', 'macAddress': '0022...",00a01427-c5ed-4721-9b3e-39bbf2a6d425,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,None,Active,NaN,Active,Active,Active
4,7f5cc4861779ea2e5fe7564f3a2b0cade08365bd,None,False,False,None,demo-win-vm,2025-07-09T15:43:14.6870664Z,2025-07-10T06:13:47.9835639Z,WindowsServer2022,x64,...,"[{'ipAddress': '10.0.1.4', 'macAddress': '6045...",ea4ce8d0-3050-467a-b619-954a6cdae7e0,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,082909d8-d042-4ef9-99f9-dd4a4ed283de,Inactive,Enabled,Inactive,Inactive,Inactive


In [7]:
# Calculate summary statistics
total_devices = len(df)
healthy_devices = df[df['Health Status'] == 'Healthy']
unhealthy_devices = df[df['Health Status'] != 'Healthy']
healthy_pct = 100 * len(healthy_devices) / total_devices if total_devices else 0
unhealthy_pct = 100 * len(unhealthy_devices) / total_devices if total_devices else 0

In [9]:
# Create donut charts for each metric
metrics = [
    ('Health Status', 'Healthy'),
    ('Sensor', 'Enabled'),
    ('Sensor Communication', 'Healthy'),
    ('Sensor Data Collection', 'Healthy'),
    ('Cloud Connectivity', 'Healthy')
]
fig = make_subplots(rows=1, cols=5, specs=[[{'type':'domain'}]*5], subplot_titles=[m[0] for m in metrics])
for i, (col, good_val) in enumerate(metrics):
    good = (df[col] == good_val).sum()
    bad = total_devices - good
    fig.add_trace(go.Pie(labels=['Healthy', 'Unhealthy'], values=[good, bad], hole=0.7,
                         marker_colors=['#21c521', '#d62728'], showlegend=False,
                         textinfo='percent',
                         hoverinfo='label+percent'),
                  1, i+1)
fig.update_layout(title_text='Device Health Overview', height=400, width=1200)
fig.show()

In [10]:
# Display summary
display(pd.DataFrame({
    'Healthy Devices': [f'{healthy_pct:.2f}% ({len(healthy_devices)} Devices)'],
    'UnHealthy Devices': [f'{unhealthy_pct:.2f}% ({len(unhealthy_devices)} Devices)']
}))

,Healthy Devices,UnHealthy Devices
0,0.00% (0 Devices),100.00% (10 Devices)


In [11]:
# Display the device table
display(df)

,id,mergedIntoMachineId,isPotentialDuplication,isExcluded,exclusionReason,computerDnsName,firstSeen,lastSeen,osPlatform,osProcessor,...,ipAddresses,vmMetadata.vmId,vmMetadata.cloudProvider,vmMetadata.resourceId,vmMetadata.subscriptionId,Health Status,Sensor,Sensor Communication,Sensor Data Collection,Cloud Connectivity
0,07db4b6b265cc224f8ddc96cd549874b889b30d2,None,False,False,None,hybrid-vm,2025-08-21T23:02:02.2462477Z,2025-08-23T06:27:06.9829603Z,WindowsServer2022,x64,...,"[{'ipAddress': '10.0.2.4', 'macAddress': '000D...",5a25b8f4-866c-445f-a490-8fc7d7df4330,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,082909d8-d042-4ef9-99f9-dd4a4ed283de,Inactive,Enabled,Inactive,Inactive,Inactive
1,18f95523fefd1962c2ff7931021e566544f6414b,None,False,False,None,testwin2,2025-06-15T12:07:44.7228143Z,2025-06-15T12:07:44.7228143Z,Windows10,None,...,"[{'ipAddress': '10.0.0.5', 'macAddress': '0022...",7a8d0984-45f4-4a7a-a755-45c450ea9aa8,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,None,Active,NaN,Active,Active,Active
2,20f5c067141911591f2929516e3db31beaddef32,None,False,False,None,test123,2025-06-23T17:00:28.090865Z,2025-06-24T06:44:36.95051Z,Ubuntu,None,...,"[{'ipAddress': '10.1.0.4', 'macAddress': '6045...",121dec4e-9cfc-40b3-9299-743303db7418,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,None,Active,NaN,Active,Active,Active
3,3b43d0758eac3d033c81bb587b09dc76e1add42b,None,False,False,None,testmdevm,2025-04-22T15:38:28.590487Z,2025-06-03T06:19:36.712736Z,Other,None,...,"[{'ipAddress': '10.0.0.4', 'macAddress': '0022...",00a01427-c5ed-4721-9b3e-39bbf2a6d425,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,None,Active,NaN,Active,Active,Active
4,7f5cc4861779ea2e5fe7564f3a2b0cade08365bd,None,False,False,None,demo-win-vm,2025-07-09T15:43:14.6870664Z,2025-07-10T06:13:47.9835639Z,WindowsServer2022,x64,...,"[{'ipAddress': '10.0.1.4', 'macAddress': '6045...",ea4ce8d0-3050-467a-b619-954a6cdae7e0,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,082909d8-d042-4ef9-99f9-dd4a4ed283de,Inactive,Enabled,Inactive,Inactive,Inactive
5,9601f128c4abaf3f41c8a37bc3938fc763f5f52a,None,False,False,None,demo-win-vm,2025-07-09T15:43:14.6870664Z,2025-07-10T06:03:47.7353591Z,WindowsServer2022,None,...,"[{'ipAddress': '10.0.1.4', 'macAddress': '6045...",ea4ce8d0-3050-467a-b619-954a6cdae7e0,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,None,Active,NaN,Active,Active,Active
6,a8a49e37d2e644ffde94b6ad2063164df1c14190,None,False,False,None,testmdevm.tr5lbxhybmzepdtwozycsfyyef.gx.intern...,2025-04-22T15:38:28.590487Z,2025-06-03T06:51:28.2532575Z,Ubuntu,x64,...,"[{'ipAddress': '10.0.0.4', 'macAddress': '0022...",00a01427-c5ed-4721-9b3e-39bbf2a6d425,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,082909d8-d042-4ef9-99f9-dd4a4ed283de,Inactive,Enabled,Inactive,Inactive,Inactive
7,b592e35b7c2fc9afd04470651fdd6e92ba1e6056,None,False,False,None,test123.nf4qzrkb1lkete2j2nbaz3af4g.gx.internal...,2025-06-23T17:00:28.090865Z,2025-06-24T06:50:51.5317685Z,Ubuntu,x64,...,"[{'ipAddress': '10.1.0.4', 'macAddress': '6045...",121dec4e-9cfc-40b3-9299-743303db7418,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,082909d8-d042-4ef9-99f9-dd4a4ed283de,Inactive,Enabled,Inactive,Inactive,Inactive
8,be7942abfa3d692a5cba1723162eccd44cfb9267,None,False,False,None,testhfh.qtttjp2bafhe1ogajc5azxgpbg.dx.internal...,2025-06-11T20:02:27.11127Z,2025-06-12T06:07:15.3550565Z,Ubuntu,x64,...,"[{'ipAddress': '10.0.0.4', 'macAddress': '6045...",7c53e8e7-ea01-46a1-b4e3-b041d632e12f,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,082909d8-d042-4ef9-99f9-dd4a4ed283de,Inactive,Enabled,Inactive,Inactive,Inactive
9,dcb6093da53ef9ad3723438d696adae6c3db232c,None,False,False,None,testhfh1.31ywzhvuhpeexm1fhebemkk1wg.dx.interna...,2025-06-11T20:02:36.713837Z,2025-06-12T18:03:18.7597165Z,Ubuntu,x64,...,"[{'ipAddress': '10.0.0.4', 'macAddress': '6045...",dfcd14bb-64a8-42f5-8914-200f5de99246,Azure,/subscriptions/082909d8-d042-4ef9-99f9-dd4a4ed...,082909